Probar con y sin ZIP y si hay que llenar los NaN de PSA con la media, 0. Mirar si hacer columna con ninguno en sintomas

In [39]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

pd.options.display.max_columns = 99999
df_train = pd.read_csv("dataset_cancer_train.csv")
df_test = pd.read_csv("dataset_cancer_test.csv")
ids = df_test['id']
df_resul=pd.DataFrame()
df_resul['Cancer']=(df_train['diagnosis']!='No Cancer').astype(int)
df_resul['diagnosis']=df_train['diagnosis']
df_resul['cancer_subtype']=df_train['cancer_subtype']
df_train = df_train.drop(columns=['cancer_subtype','diagnosis','cancer_stage'])


def procesar_symptoms(df):

    df = df.copy()

    # Inicializamos un conjunto vacío para almacenar síntomas únicos
    unique_symptoms = set()

    # Iteramos sobre cada fila para detectar nuevos síntomas
    for symptoms in df['symptoms'].dropna():
        symptoms_list = [symptom.strip() for symptom in symptoms.split(',')]
        unique_symptoms.update(symptoms_list)  # Agregamos nuevos síntomas al conjunto

    # Creamos columnas dinámicamente para cada síntoma único
    for symptom in unique_symptoms:
        df[symptom] = df['symptoms'].apply(lambda x: 1 if pd.notna(x) and symptom in x else 0)

    # Eliminamos la columna original
    df = df.drop(columns=['symptoms'])

    return df


def transformar_datos(df):

    df = df.copy()  # Evitamos modificar el DataFrame original directamente

    # Eliminamos columnas irrelevantes
    df = df.drop(columns=['id', 'employment', 'education', 'hospital_code','insurance_provider',
                          'favorite_color', 'marital_status','doctor'])
    df=procesar_symptoms(df)

    # Codificación binaria para 'sexo_paciente'
    df['sexo_paciente'] = df['sexo_paciente'].str[0].replace({'F': 0, 'M': 1})

    # Lista fija de subtipos de cáncer, excluyendo 'No cáncer'
    cancer_subtypes = [
        'Ductal Carcinoma', 'Lobular Carcinoma', 'Gastrointestinal Stromal',
        'Small Cell', 'Triple Negative', 'Inflammatory', 'Acral Lentiginous',
        'Small Cell Carcinoma', 'Nodular', 'Adenocarcinoma', 'Superficial Spreading',
        'Carcinoid', 'Lentigo Maligna', 'Neuroendocrine Tumors', 'Non-Small Cell', 'Squamous Cell'
    ]

    # Extraer el tipo ABO y el factor Rh
    df['ABO'] = df['blood_type'].str.extract(r'^(A|B|AB|O)')
    df['Rh'] = df['blood_type'].str.extract(r'([+-])$')

    # One-hot encoding para tipo ABO
    df['A'] = df['ABO'].apply(lambda x: 1 if x == 'A' else 0)
    df['B'] = df['ABO'].apply(lambda x: 1 if x == 'B' else 0)
    df['AB'] = df['ABO'].apply(lambda x: 1 if x == 'AB' else 0)
    df['O'] = df['ABO'].apply(lambda x: 1 if x == 'O' else 0)

    # Codificación binaria para Rh (1 si es positivo, 0 si es negativo)
    df['RH'] = df['Rh'].apply(lambda x: 1 if x == '+' else 0)

    # Eliminar columnas intermedias si no se requieren
    df.drop(columns=['ABO', 'Rh'], inplace=True)

    df['zip_code']=df['zip_code']/100000

    # Procesamiento de la fecha de diagnóstico
    df['diagnosis_date_paciente'] = pd.to_datetime(df['diagnosis_date_paciente'], errors='coerce')
    df['diagnosis_year'] = df['diagnosis_date_paciente'].dt.year
    df['diagnosis_age'] = df['diagnosis_year'] - df['birth_year']

    # One-hot encoding para 'medications'
    medications_list = ['Inmunoterapia', 'Quimioterapia', 'Ibuprofeno', 'Paracetamol', 'Tamoxifeno', 'Metformina']
    df_medications = pd.get_dummies(df['medications'])
    df_medications = df_medications[[col for col in df_medications.columns if col in medications_list]]
    df = pd.concat([df, df_medications.astype(int)], axis=1)
    df['PSA']=df['PSA'].fillna(0)

    # Eliminamos la columna original
    df = df.drop(columns=['hospital','blood_type','diagnosis_date_paciente', 'birth_year','diagnosis_year',
                          'medications','comorbidities'])
    df = df.reindex(sorted(df.columns), axis=1)


    return df


# Aplicar la función al DataFrame df_train
df_train = transformar_datos(df_train)
df_test=transformar_datos(df_test)
df_train
#print(df_train.columns)
#print(df_train.isna().sum().sort_values(ascending=False).head(10))


<ipython-input-39-4ed0d87775e8>:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sexo_paciente'] = df['sexo_paciente'].str[0].replace({'F': 0, 'M': 1})
<ipython-input-39-4ed0d87775e8>:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sexo_paciente'] = df['sexo_paciente'].str[0].replace({'F': 0, 'M': 1})


,A,AB,ALK,AR,B,BRAF,Bulto,C-KIT,Dificultad respiratoria,Dolor abdominal,Dolor de cabeza,Dolor en el pecho,EGFR,ER,Fatiga,Fiebre,HER2,Ibuprofeno,Inmunoterapia,KRAS,MSI,Mareo,Metformina,O,PD-L1,PR,PSA,Paracetamol,Pérdida de peso,Quimioterapia,RH,Sangrado,Sudoración nocturna,Tamoxifeno,Tos persistente,diagnosis_age,edad_paciente,sexo_paciente,zip_code
0,0,0,0.92,0.11,0,0.46,0,0.24,0,0,0,1,0.45,0,0,0,0,0,0,0.38,0.51,0,0,1,0.43,1,0.0,1,0,0,1,0,0,0,0,48,49,1,0.61175
1,1,0,0.54,0.44,0,0.32,1,0.80,0,0,0,0,0.88,0,0,0,0,0,1,0.25,0.41,0,0,0,0.96,1,0.0,0,0,0,0,0,0,0,0,75,80,1,0.21851
2,0,0,0.12,0.37,0,0.12,0,0.99,1,0,0,0,0.15,1,0,1,1,0,0,0.57,0.84,0,0,1,0.70,1,0.0,1,1,0,0,0,0,0,0,30,37,0,0.23228
3,0,0,0.14,0.18,0,0.97,0,0.44,0,0,0,0,0.40,1,0,1,0,1,0,0.73,0.28,0,0,1,0.78,1,0.0,0,0,0,0,0,0,0,1,50,56,1,0.74181
4,1,0,0.89,0.63,0,0.17,0,0.96,0,0,1,0,0.75,0,1,0,1,1,0,0.82,0.88,0,0,0,0.95,1,0.0,0,0,0,1,0,0,0,0,49,54,0,0.43703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,0,0,0.33,0.26,1,0.91,0,0.72,0,0,0,0,0.83,1,0,0,0,0,0,0.96,0.98,0,1,0,0.82,1,0.0,0,0,0,0,1,0,0,1,87,88,0,0.35877
3996,0,0,0.62,0.57,0,0.47,0,0.09,0,0,0,0,0.74,1,1,0,0,0,0,0.34,0.15,0,1,1,0.54,0,0.0,0,0,0,1,0,0,0,1,73,75,0,0.32501
3997,1,0,0.04,0.10,0,0.47,0,0.78,0,1,0,0,0.89,1,0,0,1,0,1,0.46,0.04,0,0,0,0.50,0,0.0,0,0,0,1,1,0,0,0,89,91,0,0.89802
3998,1,0,0.51,0.51,0,0.22,1,0.86,0,1,0,0,0.09,0,0,0,0,0,0,0.78,0.22,0,0,0,0.98,1,0.0,0,0,1,1,0,0,0,0,42,49,0,0.28368


In [40]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from joblib import dump, load  # Para guardar y cargar el modelo

# 📌 1. Cargar el datasheet tratado
df = df_train

# 📌 2. Definir las características (X) y la variable objetivo (y)
X = df # Excluye la columna objetivo
y = df_resul['Cancer']  # Variable binaria (0: No, 1: Sí)
y1 = df_resul['diagnosis']  # Variable multiclase en texto
y2 = df_resul['cancer_subtype']  # Variable multiclase en texto

# 📌 3. Preprocesamiento de datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir los datos en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test, y1_train, y1_test, y2_train, y2_test = train_test_split(X_scaled, y, y1, y2, test_size=0.2, random_state=42)

# 📌 4. Definir el modelo de red neuronal
model_Cancer = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Capa de entrada
    Dense(32, activation='relu'),  # Capa oculta 1
    Dense(16, activation='relu'),  # Capa oculta 2
    Dense(1, activation='sigmoid')  # Capa de salida binaria
])

# 📌 5. Compilar el modelo
model_Cancer.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 📌 6. Entrenar el modelo
model_Cancer.fit(X_train, y_train, epochs=50, batch_size=16, validation_data=(X_test, y_test))

# 📌 7. Evaluar el modelo
loss, accuracy = model_Cancer.evaluate(X_test, y_test)
print(f"\n✅ Precisión del modelo: {accuracy:.4f}")

# 📌 8. Guardar el modelo entrenado
model_Cancer.save("modelo_cancer.h5")  # Guarda el modelo en formato HDF5
dump(scaler, "scaler.joblib")  # Guarda el preprocesador de datos

# -----------------------------------------------
# 📌 9. Predicción de nuevos datos
# -----------------------------------------------
# Cargar un nuevo datasheet tratado
df_nuevos = df_test

# Preprocesar los nuevos datos con el scaler entrenado
X_nuevos = df_nuevos
X_nuevos_scaled = load("scaler.joblib").transform(X_nuevos)

# Cargar el modelo entrenado
modelo_cargado = tf.keras.models.load_model("modelo_cancer.h5")

# Realizar predicciones
predicciones_Cancer = modelo_cargado.predict(X_nuevos_scaled)


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8083 - loss: 0.5097 - val_accuracy: 0.8425 - val_loss: 0.4350
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8265 - loss: 0.4350 - val_accuracy: 0.8425 - val_loss: 0.4339
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8195 - loss: 0.4179 - val_accuracy: 0.8425 - val_loss: 0.4330
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8346 - loss: 0.3748 - val_accuracy: 0.8400 - val_loss: 0.4362
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8324 - loss: 0.3745 - val_accuracy: 0.8400 - val_loss: 0.4435
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8416 - loss: 0.3467 - val_accuracy: 0.8350 - val_loss: 0.4477
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8636 - loss: 0.3057 - val_accuracy: 0.8313 - val_loss: 0.4590
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8651 - loss: 0.3018 - val_accuracy: 0.8350 - val_


✅ Precisión del modelo: 0.7575


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [ ]:
# 📌 4. Definir el modelo de red neuronal
model_diagnosis = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Capa de entrada
    Dense(32, activation='relu'),  # Capa oculta 1
    Dense(16, activation='relu'),  # Capa oculta 2
    Dense(len(y1.unique()), activation='softmax')  # Capa de salida multiclase
])

# 📌 5. Compilar el modelo
model_diagnosis.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 📌 6. Convertir etiquetas de diagnóstico en categorías
# Ensure y1_train and y1_test have the same categories
categorias_diagnosis = pd.Categorical(pd.concat([y1_train, y1_test])) # Use pd.concat to combine Series
y_train_cod = categorias_diagnosis.codes[:len(y1_train)]  # Use appropriate indices to split
y_test_cod = categorias_diagnosis.codes[len(y1_train):]

# 📌 7. Entrenar el modelo
model_diagnosis.fit(X_train, y_train_cod, epochs=50, batch_size=16, validation_data=(X_test, y_test_cod))

# 📌 8. Evaluar el modelo
loss, accuracy = model_diagnosis.evaluate(X_test, y_test_cod)
print(f"\n✅ Precisión del modelo: {accuracy:.4f}")

# 📌 9. Guardar el modelo entrenado
model_diagnosis.save("modelo_diagnosis.h5")  # Guarda el modelo en formato HDF5
dump(categorias_diagnosis, "categorias_diagnosis.joblib")  # Guarda las categorías originales

# -----------------------------------------------
# 📌 10. Predicción de nuevos datos y guardado en DataFrame
# -----------------------------------------------
# Cargar el modelo entrenado
modelo_cargado_diagnosis = tf.keras.models.load_model("modelo_diagnosis.h5")

# Realizar predicciones
predicciones_diagnosis = modelo_cargado_diagnosis.predict(X_nuevos_scaled)

# Convertir predicciones a nombres originales de diagnóstico
categorias_diagnosis = load("categorias_diagnosis.joblib")  # Cargar categorías originales

Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# 📌 4. Definir el modelo de red neuronal
model_cancer_subtype = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Capa de entrada
    Dense(32, activation='relu'),  # Capa oculta 1
    Dense(16, activation='relu'),  # Capa oculta 2
    Dense(len(y2.unique()), activation='softmax')  # Capa de salida multiclase
])

# 📌 5. Compilar el modelo
model_cancer_subtype.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 📌 6. Convertir etiquetas de diagnóstico en categorías
# Ensure y2_train and y2_test have the same categories
categorias_cancer_subtype = pd.Categorical(pd.concat([y2_train, y2_test])) # Use pd.concat to combine Series
y2_train_cod = categorias_cancer_subtype.codes[:len(y2_train)]  # Use appropriate indices to split
y2_test_cod = categorias_cancer_subtype.codes[len(y2_train):]

# 📌 7. Entrenar el modelo
model_cancer_subtype.fit(X_train, y2_train_cod, epochs=50, batch_size=16, validation_data=(X_test, y2_test_cod))

# 📌 8. Evaluar el modelo
loss, accuracy = model_cancer_subtype.evaluate(X_test, y2_test_cod)
print(f"\n✅ Precisión del modelo: {accuracy:.4f}")

# 📌 9. Guardar el modelo entrenado
model_cancer_subtype.save("modelo_cancer_subtype.h5")  # Guarda el modelo en formato HDF5
dump(categorias_cancer_subtype, "categorias_cancer_subtype.joblib")  # Guarda las categorías originales

# -----------------------------------------------
# 📌 10. Predicción de nuevos datos y guardado en DataFrame
# -----------------------------------------------
# Cargar el modelo entrenado
modelo_cargado_cancer_subtype = tf.keras.models.load_model("modelo_cancer_subtype.h5")

# Realizar predicciones
predicciones_cancer_subtype = modelo_cargado_cancer_subtype.predict(X_nuevos_scaled)

# Convertir predicciones a nombres originales de diagnóstico
categorias_cancer_subtype = load("categorias_cancer_subtype.joblib")  # Cargar categorías originales

In [ ]:
resultados=pd.DataFrame()
resultados['id'] = ids
resultados['has_cancer']=(predicciones_Cancer > 0.5)
resultados['type']=categorias_diagnosis.categories[predicciones_diagnosis.argmax(axis=1)]
resultados['subtype']=categorias_cancer_subtype.categories[predicciones_cancer_subtype.argmax(axis=1)]
resultados.to_csv('resultados_Cancer.csv', index=False)

### Codigo de prueba

Comprobar modelo con y sin ZIP